# Replica of the scITD Analysis

The goal of this ntoebook is to replicate, on the breast-cancer data which is provided, the analysis which scITD conducts in order to gain insight on how tensor decompositions can be used to meaningfully analyse cell-cycle data.

## 0. Imports:

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

import matplotlib.pyplot as plt
import seaborn as sns

import tensorly as tl
from tensorly import tenalg

---

## 1. Data import and preprocessing:

### 1.1 Data import:

In [ ]:
adata = sc.read_h5ad("../data/breast_cancer_zikry_wayne/T47D_merged.h5ad")

Removing any protein containing $``\texttt{pRB}''$ and $``\texttt{cyto}''$:

In [ ]:
filter = [f for f in adata.var_names if ("pRB" not in f) and ("cyto" not in f)]
adata = adata[:, filter]

Understanding our data:

In [ ]:
print(f'n_obs x n_vars: {adata.shape}')

In [ ]:
print(f'cell barcodes (row index): {adata.obs_names}\n')
print(f'gene/feature names (column index): {adata.var_names}\n')

In [ ]:
adata.obs

In [ ]:
total = adata.obs['drug_name'].value_counts().sum()
adata.obs['drug_name'].value_counts() / total

In [ ]:
pd.DataFrame(adata.X, columns=adata.var_names, index=adata.obs_names)

### 1.2 Preprocessing:

We exclude phases G0 and M from our data set:

In [ ]:
condition_G0_M = (~adata.obs['GMM_phase_labels'].isin(['G0', 'M']))
adata = adata[condition_G0_M]

We subset to the following proteins:

In [ ]:
features = [
    'nuclear_mean_Ki67', 'nuclear_area', 'nuclear_mean_p16', 
    'nuclear_mean_CDH1', 'nuclear_mean_cycA2', 'nuclear_mean_cycE1',
    'nuclear_mean_CDK2', 'nuclear_mean_RB', 'nuclear_mean_CycB1',
    'nuclear_mean_p21', 'nuclear_mean_cycD1', 'nuclear_mean_CDK6',
    'nuclear_mean_CDK4', 'nuclear_mean_cycA1', 'nuclear_mean_PCNA', 
    'nuclear_mean_p53', 'nuclear_mean_p27', 'nuclear_mean_cycB2', 
    'nuclear_mean_SKP2', 'nuclear_mean_CDT1', 'nuclear_mean_cycE2',
    'DNA_int'
]

adata = adata[:, features]

In [ ]:
treatments = list(adata.obs["drug_name"].cat.categories)
phases = list(adata.obs["GMM_phase_labels"].cat.categories)
proteins = [f.replace("nuclear_mean_", "") for f in adata.var_names]

In [ ]:
import phate

# PHATE on sampled data
adata_sub = sc.pp.subsample(adata, n_obs=min(10000, adata.n_obs), copy=True, random_state=0)
phate_op = phate.PHATE()
data_phate = phate_op.fit_transform(adata_sub.X)
dominant_protein = adata_sub.to_df().idxmax(axis=1)

phate = pd.DataFrame(
    data_phate, 
    columns=["PHATE 1", "PHATE 2"],
    index=adata_sub.obs_names,
)

phate["treatment"] = adata_sub.obs["drug_name"].astype(str).values
phate["phase"] = adata_sub.obs["GMM_phase_labels"].astype(str).values
phate["protein"] = np.asarray(adata_sub[:, "DNA_int"].X).ravel()
phate["dominant_protein"] = dominant_protein.str.removeprefix("nuclear_mean_").values

In [ ]:
# Set up visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes = axes.flatten()
sns.despine(top=True, right=True, left=True, bottom=True)

dot_size = 5


# Treatment
sns.scatterplot(
    data=phate, x="PHATE 1", y="PHATE 2", hue="treatment",
    s=dot_size, linewidth=0, alpha=0.8, ax=axes[0]
)
axes[0].set_title("Treatment")
axes[0].legend(title="Treatment", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)


# Cell-cycle phase
sns.scatterplot(
    data=phate, x="PHATE 1", y="PHATE 2", hue="phase",
    s=dot_size, linewidth=0, alpha=0.8, ax=axes[1]
)
axes[1].set_title("Cell-cycle phase")
axes[1].legend(title="Phase", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)


# Dominant protein per sample
sns.scatterplot(
    data=phate, x="PHATE 1", y="PHATE 2", hue="dominant_protein",
    s=dot_size, linewidth=0, alpha=0.8, ax=axes[2], palette="tab20"
)
axes[2].set_title("Dominant protein")
axes[2].legend(
    title="Dominant protein", bbox_to_anchor=(1.02, 1),
    loc="upper left", borderaxespad=0
)

# axes[2].get_legend().remove()  # Optionally remove dominant protein legend if too long


# Final touches
for ax in axes:
    if ax == axes[0]: 
        ax.set_xlabel("PHATE 1")
        ax.set_ylabel("PHATE 2")
    else: 
        ax.set_xlabel("")
        ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

PHATE embeds each cell into two dimensions that preserve the manifold structure of the raw z-scored protein measurements, so nearby points have similar protein profiles. Treatment shows how strongly each drug separates cells in protein space; cell-cycle phase shows the variation across G1 to S to G2 progression; dominant protein labels each cell by its highest-reading marker.

---

## 2. Data analysis:

We model our tensor via: $\textit{treatment} \times \textit{cell-cycle phase} \times \textit{protein}$. Let $t \in \mathbb{N}$ be the number of treatments, in our case $t = 8$, let $c \in \mathbb{N}$ the number of cell-cycle phases, here $c = 3$, and let $p \in \mathbb{N}$ be the number of proteins analyzed, here $p = 22$. After performing a Tucker decomposition on said tensor, we may obtain: 

$$
\begin{split}
X & \approx \mathcal{G} \times_1 T \times_2 C \times_3 P \\
        & = (\mathcal{G} \times_1 T \times_2 C) \times_3 P \\
        & = \mathcal{F} \times_3 P  
\end{split}
$$

where $T \in \mathbb{R}^{t_F \times t}$ are the treatment loadings, $C \in \mathbb{R}^{c_F \times c}$ are the cell-cycle-phase loadings, $P \in \mathbb{R}^{p_F \times p}$ are the protein loadings, and $\mathcal{F} \in \mathbb{R}^{t \times c \times p_F}$ is a tensor encoding the effect of the treatments on the protein-factors across cell-cycles. (Note that $t_F$, $c_F$, and $p_F$ denote the treatment-facotrs, cell-cycle-phase-factors, and protein-factors, respectively.)

### 2.1 Pseudobulking:

Similar to Mitchel et al. (2025), we perform pseudobulking, averaging proteins within each $\texttt{(treatment, cell-cycle-phase)}$ and obtaining our tensor by stacking across proteins.

In [ ]:
def pseudobulk(adata):
    # Convert AnnData to DataFrame 
    adata_df = adata.to_df()
    
    # Group by drug_name and GMM_phase_labels, then get mean of each group
    pseudobulk_df = adata_df.groupby([adata.obs["drug_name"], adata.obs["GMM_phase_labels"]], observed=True).mean()
    
    # z-scale 
    pseudobulk_df_scales = (pseudobulk_df - pseudobulk_df.mean()) / pseudobulk_df.std()
    
    # Reshape to (n_treatments, n_phases, n_proteins)
    return_tensor = pseudobulk_df_scales.values.reshape(len(adata.obs["drug_name"].cat.categories), len(adata.obs["GMM_phase_labels"].cat.categories), -1)
    return return_tensor

In [ ]:
pseudobulk_tensor = pseudobulk(adata)
print(f'Pseudobulk tensor shape: {pseudobulk_tensor.shape}')

print("Example pseudobulk values for the first protein:")
pd.DataFrame(pseudobulk_tensor[:, :, 0], index=adata.obs["drug_name"].cat.categories, columns=adata.obs["GMM_phase_labels"].cat.categories)

In [ ]:
def HOSVD(tensor, truncation_rank=None):
    dims = tensor.shape
    
    if truncation_rank is None:
        truncation_rank = dims
    
    factors = []
    for i, _ in enumerate(dims): 
        matrix = tl.unfold(tensor, mode=i)
        
        rank = np.linalg.matrix_rank(matrix)
        factor = np.linalg.svd(matrix, full_matrices=False)[0][:, :rank]
        
        factors.append(factor[:, :truncation_rank[i]])
    
    factors_transposed = [f.T for f in factors]
    
    core = tenalg.multi_mode_dot(tensor, factors_transposed)
        
    return core, factors

### 2.2 Protein Rank-Selection:

We consider first the protein rank. To do so, we compute Tucker decompositions of our tensor, keeping the ranks of the other axes fixed at the maximum. 

In [ ]:
err_dict = {}

for i in range(len(proteins)):
    truncation_rank = [pseudobulk_tensor.shape[0], pseudobulk_tensor.shape[1], i+1]
    core, factors = HOSVD(pseudobulk_tensor, truncation_rank=truncation_rank)
    
    recon = tenalg.multi_mode_dot(core, factors)
    err = np.linalg.norm(recon - pseudobulk_tensor) / np.linalg.norm(pseudobulk_tensor)
    
    err_dict[i+1] = err

We interpolate the first an the last value and compute the difference between the reconstruction loss and the interpolation line. The value to be chosen is that which maximizes the difference. 

In [ ]:
start_err = err_dict[1]
end_err = err_dict[len(err_dict)]
steps = len(err_dict)

interpol_line = {i: (start_err * (((steps-1) - (i-1))/(steps-1)) + end_err * ((i-1)/(steps-1))) for i in range(1, steps+1)}
diff = {i: interpol_line[i] - err_dict[i] for i in range(1, steps+1)}

n_protein_factors = max(diff, key=diff.get)
print(f"Maximum difference at step {n_protein_factors}: {diff[n_protein_factors]}")

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1)

ax.set_ylabel("Reconstruction Error")
ax.set_xlabel("Protein Rank")
ax.set_xticks([k for k in err_dict if k % 4 == 0])

ax.spines[["top", "right"]].set_visible(False)

ax.plot(list(err_dict.keys()), list(err_dict.values()), marker='o')
ax.plot(list(interpol_line.keys()), list(interpol_line.values()), alpha=0.7, marker='x', linestyle='--')
ax.plot(list(diff.keys()), list(diff.values()), alpha=0.7, marker='^', linestyle='--')
ax.vlines(n_protein_factors, 0, err_dict[1], colors='red', alpha=0.25, linestyle='--')

plt.legend(["Reconstruction Error", "Interpolated", "Difference"])
plt.title("Reconstruction Error vs. Protein Rank")
plt.show()

In [ ]:
truncation_rank = [pseudobulk_tensor.shape[0], pseudobulk_tensor.shape[1], n_protein_factors]
core, factors = HOSVD(pseudobulk_tensor, truncation_rank=truncation_rank)

treatment_loadings = factors[0]
cell_phase_loadings = factors[1]
protein_loadings = factors[2]

---

## 3. Visualizations:

### 3.1 Latent Protein-Factors over Tensor Components:

In [ ]:
from numpy import matrix


def matrix_heatmap(matrix, xticks=None, yticks=None, xlabel=None, ylabel=None, title=None, x_tick_size=8, y_tick_size=8, text_size=8):
    fig = plt.figure(figsize=(16, 14))
    ax = fig.add_subplot(1, 1, 1)

    vmax = np.abs(matrix).max() or 1.0

    im = ax.imshow(matrix, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")

    if xticks is not None:
        ax.set_xticks(np.arange(len(xticks)))
        ax.set_xticklabels(xticks, fontsize=x_tick_size)
    else:
        ax.set_xticks(np.arange(matrix.shape[1]))
        ax.set_xticklabels(np.arange(1, matrix.shape[1] + 1))

    if yticks is not None:
        ax.set_yticks(np.arange(len(yticks)))
        ax.set_yticklabels(yticks, fontsize=y_tick_size)
    else: 
        ax.set_yticks(np.arange(matrix.shape[0]))
        ax.set_yticklabels(np.arange(1, matrix.shape[0] + 1))
    
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=10)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=10)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(
                j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=text_size,
                color="white" if abs(matrix[i, j]) > 0.6 * vmax else "black"
            )
                
    fig.colorbar(im, ax=ax, orientation="horizontal", shrink=0.4, pad=0.05)

    if title:
        plt.title(title, fontsize=13)
        
    plt.show()

In [ ]:
matrix_heatmap(
    protein_loadings, 
    yticks=proteins,
    xlabel="Latent Protein-Factors", ylabel="Protein", 
    title="Protein Loadings", 
    y_tick_size=8, text_size=8
)

Each column is one latent protein-factor, an orthonormal direction in protein space; each row is a protein's weight on those directions. Values are to be interpreted contrastively; proteins with the same sign on a factor co-vary along it, opposite signs move in opposition. (Note that the overall sign of a column is arbitrary, since flipping a factor and its matching scores leaves the reconstruction unchanged, so read the pattern of signs within a column rather than the absolute sign.) 

Because the pseudobulk is z-scored, the overall intensity doesn't influence the individual loadings. Values capture contrast between proteins. 

We now also consider the tensor $\mathcal{F} = \mathcal{G} \times_1 T \times_2 C \in \mathbb{R}^{t \times c \times p_F}$. We may obtain matrices $\mathbf{F}^{(T)} \in \mathbb{R}^{p_F \times t}$, and $\mathbf{F}^{(C)} \in \mathbb{R}^{p_F \times c}$ by averaging over the remaining dimension. 

In [ ]:
F = tenalg.multi_mode_dot(
    core,
    [treatment_loadings, cell_phase_loadings],
    modes=[0, 1],
)

**Latent Protein-Factors by treatment:**

In [ ]:
proteins_treatment_loadings = F.mean(axis=1).T  # average over: (1) cell-cycle phases
proteins_treatment = proteins_treatment_loadings @ treatment_loadings.T

matrix_heatmap(
    proteins_treatment, 
    xticks=treatments,
    xlabel="Treatment", ylabel="Latent Protein-Factors", 
    title="Protein-Factors x Treatment (Averaged over Cell Phases)"
)

F averaged over cell-cycle phase, then mapped back onto the treatments through the treatment loadings, giving each latent protein-factor's average level per drug. A positive entry means that drug pushes the factor in its positive direction relative to other treatments; one would have to combine the sign here with the protein loadings to see which proteins rise or fall. Columns that look alike flag drugs with similar multiprotein effects.

**Latent Protein-Factors by cell-cycle phase:**

In [ ]:
proteins_phase_loadings = F.mean(axis=0).T  # average over: (0) treatments
proteins_phase = proteins_phase_loadings @ cell_phase_loadings.T

matrix_heatmap(
    proteins_phase, 
    xticks=phases, 
    xlabel="Cell phase", ylabel="Latent Protein-Factors", 
    title="Protein-Factors x Cell Phase (Averaged over Treatments)"
)

Same construction averaged over treatments, showing how each latent protein-factor varies across G1, S, and G2. This shows how cell-cycle influences each multiprotein factor; a factor whose sign flips across phases encodes a phase-specific program. 

### 3.2 Tensor Slices for Different Cell-Lines:

In [ ]:
def slice_montage(F, treatments, phases, annotate=True):
    n_phases = F.shape[1]
    n_factors = F.shape[2]
    vmax = np.abs(F).max() or 1.0   # shared scale across all phases

    fig, axes = plt.subplots(
        1, n_phases, figsize=(8 * n_phases, 6), squeeze=False
    )
    axes = axes.ravel()

    for c in range(n_phases):
        ax = axes[c]
        im = ax.imshow(
            F[:, c, :], cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto"
        )
        ax.set_title(f"{phases[c]}", fontsize=10)
        ax.set_xticks(np.arange(n_factors))
        ax.set_xticklabels(np.arange(1, n_factors + 1), fontsize=8)
        ax.set_xlabel("Latent Protein-Factor", fontsize=8)
        ax.set_yticks(np.arange(len(treatments)))
        ax.set_yticklabels(treatments if c == 0 else [], fontsize=8)

        if annotate:
            for i in range(F.shape[0]):
                for j in range(n_factors):
                    v = F[i, c, j]
                    ax.text(
                        j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if abs(v) > 0.6 * vmax else "black",
                    )

    fig.suptitle("F — Treatment x Latent Protein-Factors by Phase", position=(0.44, 1.0))
    fig.colorbar(im, ax=axes.tolist(), orientation="vertical", shrink=0.8, pad=0.02)
    plt.show()

In [ ]:
slice_montage(F, treatments, phases, annotate=True)

Each panel is one cell-cycle phase slice of F (slices are: $\textit{treatment} \times \textit{latent protein-factor}$). Reading a single treatment's row across the panels shows how its factor profile changes with phase; reading down a column compares treatments within one phase. This shows the actual values of $F$ before the averaged treatment heatmaps above.

---

## 4. PHATE with Latent-Protein Factors:

We may project our original data matrix $\mathbf{X} \in \mathbb{R}^{(n \times p)}$ into the latent protein-factor space by: $\mathbf{X} \cdot \mathbf{P} \in \mathbb{R}^{n \times p_F}$, where $\mathbf{P} \in \mathbb{R}^{p \times p_F}$ are our protein-factor loadings. 

In [ ]:
X = adata_sub.X
X = (X - X.mean(0)) / X.std(0)

embed = pd.DataFrame(X @ protein_loadings, columns=[f"Factor_{i+1}" for i in range(protein_loadings.shape[1])])

In [ ]:
n_factors = embed.shape[1]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
sns.despine(top=True, right=True, left=True, bottom=True)

for k, ax in enumerate(axes):
    if k >= n_factors:
        ax.axis("off")
        continue

    c = embed.iloc[:, k]
    vmax = np.nanpercentile(np.abs(c), 99) or 1.0
    pts = ax.scatter(
        data_phate[:, 0], data_phate[:, 1], c=c, cmap="RdBu_r",
        vmin=-vmax, vmax=vmax, s=dot_size, linewidth=0, alpha=0.8,
    )
    ax.set_title(f"Factor {k + 1}")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(pts, ax=ax, shrink=0.7, pad=0.02)

plt.tight_layout()
plt.show()

### Dominant Latent Protein-Factor per Treatment

We split the shared PHATE embedding by treatment and color each cell by the latent protein-factor on which it scores highest in absolute value. 

In [ ]:
from matplotlib.lines import Line2D

# Dominant latent protein-factor per cell (largest absolute score; factors are signed contrasts)
dominant_factor = embed.abs().idxmax(axis=1).values  # positional, aligned to data_phate
factor_names = list(embed.columns)
palette = dict(zip(factor_names, sns.color_palette("tab10", len(factor_names))))

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()
sns.despine(top=True, right=True, left=True, bottom=True)

treat_vals = phate["treatment"].values  # positional, aligned to data_phate
for a, treatment in enumerate(treatments):
    ax = axes[a]
    mask = treat_vals == treatment
    ax.scatter(
        data_phate[mask, 0], data_phate[mask, 1],
        c=[palette[f] for f in dominant_factor[mask]],
        s=dot_size, linewidth=0, alpha=0.8
    )
    ax.set_title(treatment)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(""); ax.set_ylabel("")

handles = [
    Line2D([0], [0], marker="o", linestyle="", color=palette[f], label=f)
    for f in factor_names
]

fig.legend(
    handles=handles, title="Dominant factor",
    bbox_to_anchor=(1.0, 0.5), loc="center left", borderaxespad=0
)
plt.tight_layout()
plt.show()

We use the "original" PHATE embedding restricted to one treatment's cells, colored by which latent protein-factor dominates that cell. If a drug shifts its cells toward a different dominant color than control, it is pushing cells into a different multiprotein program. Factors are signed, so dominance here is by absolute score; a cell colored by a given factor may sit at either the positive or negative end of it.

---